## Load and clean the CEADRIZ study clinical and sequencing data

In [31]:
import os
import re
import glob
import datetime
import pandas as pd
import numpy as np
import seaborn as sns
import scipy.stats as ss
import statsmodels.formula.api as smf
from dataclasses import dataclass

import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpec

## Settings

In [32]:
DIR_SEQDATA = "../seqdata/exvivo/summaries/exvivo" # could also go from here
DIR_DAY3DATA = "../day3data/"
DIR_FIGS = "../figures/fig4"
DIR_TABLES = "../tables"
os.makedirs(DIR_FIGS, exist_ok=True)
save_results = True
W = 3.4 # roughly width of one column

In [33]:
P = lambda n, N: f"{n} ({100*n/N:.2f}%) of {N}"

## Load clinical data

In [34]:
def clean_parasite_density(pul: pd.Series) -> pd.Series:
    """
    Reprocess a parasite density column to make it numeric, by:
    NMPS = 0
    - = None
    """
    result = []
    for p in pul:
        if p == 'NMPS':
            result.append(0)
            continue
        if p == "_":
            result.append(None)
            continue
        if isinstance(p, int) or isinstance(p, float):
            result.append(p)
            continue

        # Finally try to pull out numbers
        nums = re.match("[0-9]+", p)
        if nums is None:
            print(f"No number found for {p}, setting as None.")
            result.append(None)
        else:
            result.append(float(nums.group(0)))
    return pd.Series(data=result).astype(float)

In [35]:
def fix_dates(date: str) -> str:
    fmts = [
        "2026-(?P<day>[0-9]{2})-(?P<month>0[45]{1})",
        "(?P<day>[0-9]{2})/(?P<month>0[45]{1})/2026"
    ]
    for fmt in fmts:
        match = re.match(fmt, date)
        if match is not None:
            break
    if match is None:
        raise ValueError(f"Could not parse {date}.")

    return f"2026-{match.group('month')}-{match.group('day')}"

In [36]:
def standardise_placenames(name):
    name = name.lower().strip()
    name = re.sub(r"[^a-z0-9\s]", "", name) # punctuation
    name = re.sub(r"\s+", "_", name) # white space
    return name

In [37]:
def define_completion_status(collection_date, qualified: str, day_3_parasite_density: float) -> str:
    """Define the completion of a patient"""
    if qualified != "yes":
        return "not_enrolled"
    if collection_date > (pd.Timestamp.now() - pd.Timedelta(days=3)):
        return "waiting"
    if pd.isna(day_3_parasite_density): # is this the best way to determine if someone was lost to follow up?
        return "lost"
    return "complete"

In [38]:
def define_day3_positive(status, day_3_parasite_density: float) -> bool:
    if status != 'complete':
        return None
    if day_3_parasite_density > 0:
        return True
    return False

In [39]:
def clean_treatment_history(history: str) -> str:
    if history == 'No':
        return 'no'
    if history.lower().startswith('y'):
        return 'yes'
    return None

In [40]:
def fix_qualifed(qualified: str, infection_rate: float, history: str) -> str:
    """
    Unfortunately, it seems a few individuals were enrolled at the clinic who
    did not meet our inclusion criterion; fix here
    """
    if qualified == 'Yes':
        if infection_rate < 0.1:
            print("Parasitemia too low to enroll.")
            return 'no'
        if history == 'yes':
            print("Had treatment history.")
            return 'no'
        return 'yes'
    return qualified.lower()

In [41]:
# Load
df_day3 = pd.read_excel("../day3data/Kaoma Local List.xlsx", 
                        skiprows=3, 
                        dtype={"Sample ID": str, "Collection Date": str},
                       )
df_day3.columns = (df_day3.columns
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("_(p/µl)", "")
    .str.replace("day_0", "day0")
    .str.replace("day_3", "day3")
    .str.replace("?", "")
)

In [42]:
df_day3["comment"] = [f"{n}" if pd.isna(u) else f"{n} | {u}" 
                      for n, u in zip(df_day3["comment"], df_day3["unnamed:_19"])]

In [43]:
# Drop unused columns here
assert all(df_day3["rdt_result"] == 'Positive')
assert all(df_day3["day0_dbs_collected"] == 'Yes')

df_day3.drop(["unnamed:_19", # added to comment
              "travel_history", # only one participant
              "day0_dbs_collected", # always positive
              "rdt_result", # always yes
              "sn", "field_id" # neither is needed here
             ], axis=1, inplace=True)

In [44]:
# Dates
df_day3['collection_date'] = pd.to_datetime(
    df_day3['collection_date'].apply(fix_dates),
    format="%Y-%m-%d"
)

# Villages
village_probable_synonyms = { # from manual inspection only
    "mulambwa": "mulamba",
    "malamba": "mulamba",
    "mulambwa": "mulamba",
    "fland": "folkland"
}
df_day3["village"] = (df_day3["village"]
    .apply(standardise_placenames)
    .replace(village_probable_synonyms)
)

In [45]:
# Parasitemia
df_day3["day0_parasite_density"] = clean_parasite_density(df_day3["day0_parasite_density"])
df_day3["day3_parasite_density"] = clean_parasite_density(df_day3["day3_parasite_density"]) 
df_day3["day0_parasite_density_log10"] = [np.log10(v) if v > 0 else None for v in df_day3["day0_parasite_density"]]
df_day3["day3_parasite_density_log10"] = [np.log10(v) if v > 0 else None for v in df_day3["day3_parasite_density"]]
df_day3["infection_rate"] = [100*float(r) if str(r).startswith('0') else None for r in df_day3.infection_rate]
# note there are zeros, so we get runtime errors

No number found for +Gametocytes               73019, setting as None.
No number found for Gametocytes only, setting as None.
No number found for Gametocytes only, setting as None.
No number found for + Gametocytes              36217, setting as None.
No number found for Gametocytes only, setting as None.
No number found for #ERROR!, setting as None.
No number found for Gametocytes only, setting as None.
No number found for Gametocytes only, setting as None.
No number found for Gametocytes only, setting as None.
No number found for Gametocytes only, setting as None.


In [46]:
# Age, sex and fever
df_day3["age"] = [float(a) if a != '_' else None for a in df_day3['age']]
df_day3["sex"] = [s if s != '_' else None for s in df_day3['sex']]
df_day3["fever"] = [a if a != '_' else None for a in df_day3['fever']]

In [47]:
# Treatment History
df_day3["treatment_history"] = [clean_treatment_history(h)
                                for h in df_day3["treatment_history"]]

In [48]:
df_day3["qualified"] = [fix_qualifed(q, p, h) 
                        for q, p, h in zip(df_day3["qualified"], 
                                           df_day3["infection_rate"], 
                                           df_day3["treatment_history"])]

Had treatment history.
Parasitemia too low to enroll.
Parasitemia too low to enroll.
Had treatment history.
Had treatment history.
Had treatment history.
Had treatment history.
Had treatment history.
Had treatment history.
Had treatment history.
Had treatment history.


In [49]:
# Sample status in study
df_day3["status"] = [define_completion_status(d, q, p) 
                     for d, q, p in zip(df_day3["collection_date"],
                                        df_day3["qualified"],
                                        df_day3["day3_parasite_density"])]

In [50]:
# Sample day 3 positivity
df_day3["day3_pos"] = [define_day3_positive(s, p)
                       for s, p in zip(df_day3["status"],
                                       df_day3["day3_parasite_density"])]

In [51]:
if save_results:
    df_day3.to_csv(f"{DIR_FIGS}/ceadriz.clinical_data.csv", index=False)

#### Examine clinical data

In [52]:
pd.crosstab(
    df_day3["qualified"],
    df_day3["status"],
    margins=True)

status,complete,lost,not_enrolled,All
qualified,,,,
no,0,0,118,118
yes,103,85,0,188
All,103,85,118,306


In [53]:
ct_status = pd.crosstab(
    df_day3["collection_date"],
    df_day3["status"],
    margins='All'
)
ct_status

status,complete,lost,not_enrolled,All
collection_date,,,,
2026-04-02 00:00:00,4,1,4,9
2026-04-04 00:00:00,9,2,16,27
2026-04-07 00:00:00,12,9,6,27
2026-04-09 00:00:00,11,3,3,17
2026-04-11 00:00:00,12,6,8,26
2026-04-13 00:00:00,13,11,7,31
2026-04-15 00:00:00,8,8,14,30
2026-04-17 00:00:00,12,14,11,37
2026-04-18 00:00:00,7,7,8,22


In [54]:
PER = lambda n,N: f"{100*n/N:.2f}"
N_all = ct_status['All']['All']
N_enrolled = N_all - ct_status['not_enrolled']['All']
N_complete = ct_status['complete']['All']
N_day3_dbs = (df_day3.loc[(df_day3['qualified'] == 'yes')]['day3_dbs_collected'] == 'Yes').sum()
N_dbs_total = N_enrolled + N_day3_dbs
print(f"""

Had a total of {N_all} visitors to clinic.
Of those, we enrolled {N_enrolled} ({PER(N_enrolled, N_all)}%)
Of those enrolled, {N_complete} ({PER(N_complete, N_enrolled)}%) returned on day 3.

We have a total of {N_dbs_total} DBS to sequence:
  Day 0: {N_enrolled}
  Day 3: {N_day3_dbs}

""")



Had a total of 306 visitors to clinic.
Of those, we enrolled 188 (61.44%)
Of those enrolled, 103 (54.79%) returned on day 3.

We have a total of 294 DBS to sequence:
  Day 0: 188
  Day 3: 106




In [55]:
ct_day3 = pd.crosstab(
    df_day3["collection_date"],
    df_day3["day3_pos"],
    margins='All',
)
ct_day3

day3_pos,False,True,All
collection_date,,,
2026-04-02 00:00:00,2,2,4
2026-04-04 00:00:00,6,3,9
2026-04-07 00:00:00,11,1,12
2026-04-09 00:00:00,9,2,11
2026-04-11 00:00:00,11,1,12
2026-04-13 00:00:00,10,3,13
2026-04-15 00:00:00,5,3,8
2026-04-17 00:00:00,11,1,12
2026-04-18 00:00:00,5,2,7


- Quite even rate of day 3 positive across collection days

In [56]:
N_pos = ct_day3[True]['All']
print(f"""
Of the {N_complete} returning on day 3, {N_pos} ({PER(N_pos, N_complete)}%) were positive.
""")


Of the 103 returning on day 3, 21 (20.39%) were positive.



In [57]:
pd.crosstab(
    df_day3["day3_dbs_collected"],
    df_day3["day3_pos"],
    margins='All',
)

day3_pos,False,True,All
day3_dbs_collected,,,
No,1,0,1
Yes,81,21,102
All,82,21,103


- We have a day 3 DBS for all of those that were positive on day 3, which is good

## Load sequencing data

In [58]:
def get_day_information(sample_id: str) -> int:
    """
    Encoded like <SAMPLE_ID>-D<DAY>, if day > 0
    """
    fields = sample_id.split("-")
    if len(fields) == 1:
        return 0
    else:
        return int(fields[1][-1])

In [59]:
@dataclass
class K13:
    sample_id: str
    a724_gt: str
    a724_wsaf: float
    a724_dp: float 
    k13_all: str # concat all k13 mutations
    k13_wsaf: float # sum all k13 mutations
    
    # classifications
    a724_mutant: str = None # mutant or wildtype
    k13_mutant: str = None # mutant or wildtype

    def __post_init__(self):        
        self.a724_mutant = {'0/0': 'wildtype',
                            '0/1': 'mutant',
                            '1/1': 'mutant',
                            './.': 'failed'}[self.a724_gt]

        # TODO: If the amplicon has failed, what happens here?
        # - Failed data is removed by `nomadic summarize`
        self.k13_mutant = 'wildtype' if self.k13_all == 'WT' else 'mutant'

In [60]:
# Load and filter to kelch13
df_kelch13_all = (
    pd.read_csv(f"{DIR_SEQDATA}/summary.variants.analysis_set.csv", dtype={"sample_id": str})
    .query("gene == 'kelch13'")
)

In [61]:
# Get a list of K13 mutations that are non-synonymous
NONSYN_K13 = (df_kelch13_all[['mut_type', 'aa_pos', 'aa_change']]
    .dropna()
    .drop_duplicates()
    .query("mut_type == 'missense'")
    .aa_pos
    .to_list()
)
print(NONSYN_K13)

[667, 724, 622, 667, 473, 485, 675, 441, 533, 637]


In [62]:
df_kelch13_focus = df_kelch13_all.query("aa_pos in @NONSYN_K13")

In [63]:
results = []
for sample_id, sdf in df_kelch13_focus.groupby("sample_id"):
    # Get information about a724, regardless
    a724 = list(sdf.query("aa_pos == 724")[["gt", "wsaf", "dp"]].values[0])

    # Get other mutation information
    mut_ix = sdf.type != 'wt'
    if not mut_ix.sum():
        results.append(K13(sample_id, *a724, k13_all="WT", k13_wsaf=0.0))
    else:
        k13_all = "-".join(sdf.loc[mut_ix, 'aa_change'])
        k13_wsaf = sdf.loc[mut_ix, 'wsaf'].sum()
        results.append(K13(sample_id, *a724, k13_all, k13_wsaf))

In [64]:
df_kelch13 = pd.DataFrame(results)

In [65]:
df_kelch13.head() # this is going to go forward, now need to annotate and make wide

,sample_id,a724_gt,a724_wsaf,a724_dp,k13_all,k13_wsaf,a724_mutant,k13_mutant
0,8033751,0/0,0.000000,813.0,P667A,1.000000,wildtype,mutant
1,8033754,1/1,0.990040,2677.0,A724E,0.990040,mutant,mutant
2,8033754-D3,0/1,0.852827,3434.0,P667A-A724E,0.879491,mutant,mutant
3,8033755,0/1,0.237340,4613.0,P667S-A724E,0.917721,mutant,mutant
4,8033756,0/0,0.000453,2079.0,WT,0.000000,wildtype,wildtype


In [66]:
df_kelch13["treatment_day"] = [get_day_information(s) for s in df_kelch13["sample_id"]]

In [67]:
df_kelch13["sample_id_d3_indicated"] = df_kelch13["sample_id"]
df_kelch13["sample_id"] = [s.split("-")[0] for s in df_kelch13["sample_id_d3_indicated"]]
df_kelch13.drop(["sample_id_d3_indicated"], axis=1, inplace=True)

In [68]:
# Day 0
_df_day0 = df_kelch13.query("treatment_day == 0").drop('treatment_day', axis=1)
_df_day0.columns = [f"day0_{c}" if c != 'sample_id' else c
                    for c in _df_day0.columns]

# Day 3
_df_day3 = df_kelch13.query("treatment_day == 3").drop('treatment_day', axis=1)
_df_day3.columns = [f"day3_{c}" if c != 'sample_id' else c
                    for c in _df_day3.columns]

In [69]:
_df_day0.shape[0], _df_day3.shape[0] # 188 on day 0, 23 on day 3

(188, 23)

In [70]:
df_kelch13_wide = pd.merge(
    left=_df_day0,
    right=_df_day3,
    on=["sample_id"],
    how="left",
    validate="1:1"
)
df_kelch13_wide.shape

(188, 15)

In [71]:
if save_results:
    df_kelch13.to_csv(f"{DIR_FIGS}/ceadriz.kelch13.csv", index=False)
    df_kelch13_wide.to_csv(f"{DIR_FIGS}/ceadriz.kelch13_wide.csv", index=False)

## Merge

In [72]:
# No sequencing IDs should be unique
set(df_kelch13_wide.sample_id).difference(df_day3.sample_id)

set()

In [73]:
# All sequencing IDs should be inside of the day 3 data
assert \
    len(set(df_kelch13_wide.sample_id).intersection(df_day3.sample_id)) == df_kelch13_wide.shape[0]

In [74]:
df_merged = pd.merge(
    left=df_day3,
    right=df_kelch13_wide,
    how='left',
    on='sample_id',
    validate='1:1'
)

In [75]:
df_merged["day0_seqdone"] = [not pd.isna(v) for v in df_merged['day0_a724_mutant']]
df_merged["day3_seqdone"] = [not pd.isna(v) for v in df_merged['day3_a724_mutant']]

In [76]:
ct_status = pd.crosstab(
    df_merged["status"],
    df_merged["day0_seqdone"],
    margins=True
)
ct_status

day0_seqdone,False,True,All
status,,,
complete,5,98,103
lost,8,77,85
not_enrolled,105,13,118
All,118,188,306


- We have 13 samples not enrolled, that were sequenced
- We have 13 samples that *were* enrolled, that failed sequencing QC

In [77]:
N_complete = ct_status.loc['complete', 'All']
N_seq = ct_status.loc['complete', True]
print(f"""
Of those returning on day 3, we have day 0 sequencing data for {P(N_seq, N_complete)}.
""")


Of those returning on day 3, we have day 0 sequencing data for 98 (95.15%) of 103.



In [78]:
samples_notseq = df_merged.query("status == 'complete' and not day0_seqdone")["sample_id"].tolist()
df_metadata = pd.concat(
    [pd.read_csv(csv, dtype=dict(sample_id=str)) 
     for csv in glob.glob("../seqdata/exvivo/results/*/metadata/samples.csv")])

In [79]:
df_metadata.query("sample_id in @samples_notseq")

,sample_id,sample_type,barcode,post_pcr_dna_conc_ngul
42,8033785,Field,barcode43,5.5000
10,8036313,Field,barcode11,7.3800
23,8036328,Field,barcode24,0.0218
35,8036349,Field,barcode36,36.0000
49,8036364,Field,barcode50,8.7600


- They were all processed, so they have failed QC

In [80]:
if save_results:
    df_merged.to_csv(f"{DIR_FIGS}/ceadriz.merged.csv", index=False)